# Preparacion de salida para carga al mosaico

Notebook minimo para validar imagenes nuevas antes de copiarlas a la estructura del datastore. La entrada es una carpeta de imagenes y el feature class de sectores. La salida es un manifiesto con el nombre esperado por cruce geografico.

In [ ]:
from datetime import datetime
from pathlib import Path
import importlib

import pandas as pd

import core.mosaic_image_audit as mosaic_audit
mosaic_audit = importlib.reload(mosaic_audit)
from core.mosaic_image_audit import *

# PARAMETROS MINIMOS
PATH_INPUT_IMAGENES = r"\\amssclgis10.ams.gmams.cl\CL_MLP_PAO\Vuelos_Drone_Sin_Procesar\INPUT\20260519_Geosupport"
PATH_FC_INDICE_VUELOS_IMGS = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"
SECTOR_FIELD_INDICE_VUELOS = "Sector"
QUERY_INDICE_VUELOS = None  # Ejemplo: "Estado = 'Activo'"

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path.cwd() / "outputs" / "preparacion_carga_mosaico" / run_timestamp
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Modulo auditoria:", mosaic_audit.__file__)
print("Version logica:", AUDIT_LOGIC_VERSION)
print("Input imagenes:", PATH_INPUT_IMAGENES)
print("Feature sectores:", PATH_FC_INDICE_VUELOS_IMGS)
print("Campo sector:", SECTOR_FIELD_INDICE_VUELOS)
print("Query sectores:", QUERY_INDICE_VUELOS)
print("Salida:", OUTPUT_DIR)

## 1. Buscar imagenes

La busqueda es recursiva. Solo se preparan `tif` y `tiff` para la carga al mosaico.

In [ ]:
input_images_df = scan_input_images(PATH_INPUT_IMAGENES)
ortho_images_df = input_images_df[input_images_df["extension"].isin(ORTHO_MOSAIC_EXTENSIONS)].copy()

print(f"Archivos encontrados: {len(input_images_df)}")
print(f"Imagenes TIF/TIFF para evaluar: {len(ortho_images_df)}")

display(input_images_df.groupby("extension").size().reset_index(name="count"))
display(ortho_images_df[["file_name", "relative_path", "size_mb", "modified_at"]].head(20))

## 2. Calcular sector geografico y nombre esperado

El sector viene solo del cruce espacial. Si una imagen cruza mas de un sector, se usa el sector con mayor porcentaje de interseccion. El texto del sector se conserva desde el feature class y solo se reemplazan espacios por `_`.

In [ ]:
spatial_matches_df = calculate_spatial_sector_matches(
    ortho_images_df,
    PATH_FC_INDICE_VUELOS_IMGS,
    sector_field=SECTOR_FIELD_INDICE_VUELOS,
    where_clause=QUERY_INDICE_VUELOS,
)

prepared_df = add_expected_names_with_spatial_sector(
    ortho_images_df,
    spatial_matches_df,
)

display(spatial_matches_df["spatial_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "spatial_status"}))
display(prepared_df["rename_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "rename_status"}))

## 3. Preparar manifiesto de salida

`ready_for_datastore` indica registros listos para validar y luego copiar. Este notebook no copia archivos.

In [ ]:
manifest_df = prepared_df.copy()

manifest_df["destination_date_folder"] = manifest_df["expected_date_token"].map(
    lambda value: "_".join(str(value).split("_")[:2]) if pd.notna(value) and value else None
)
manifest_df["ready_for_datastore"] = manifest_df["rename_status"].eq("ok")
manifest_df["duplicate_expected_file_name"] = (
    manifest_df["expected_file_name"].notna()
    & manifest_df.duplicated("expected_file_name", keep=False)
)

def build_review_reason(row):
    reasons = []
    if row.get("rename_status") != "ok":
        reasons.append(str(row.get("rename_status")))
    if row.get("duplicate_expected_file_name"):
        reasons.append("nombre_esperado_duplicado")
    if row.get("spatial_overlap_count", 0) and row.get("spatial_overlap_count", 0) > 1:
        reasons.append("cruza_multiples_sectores")
    return "|".join(reasons) if reasons else None

manifest_df["review_reason"] = manifest_df.apply(build_review_reason, axis=1)

output_columns = [
    "ready_for_datastore",
    "review_reason",
    "path",
    "relative_path",
    "file_name",
    "expected_file_name",
    "expected_name",
    "expected_date_token",
    "destination_date_folder",
    "expected_sector",
    "sector_source",
    "rename_status",
    "spatial_status",
    "spatial_sector_raw",
    "spatial_sector",
    "spatial_overlap_pct",
    "spatial_overlap_count",
    "spatial_all_matches",
    "duplicate_expected_file_name",
    "size_mb",
    "modified_at",
]
output_columns = [column for column in output_columns if column in manifest_df.columns]
manifest_output_df = manifest_df[output_columns].copy()

print(f"Listas para validar/copiar: {int(manifest_output_df['ready_for_datastore'].sum())}")
print(f"Requieren revision: {int((~manifest_output_df['ready_for_datastore']).sum())}")
print(f"Nombres esperados duplicados: {int(manifest_output_df['duplicate_expected_file_name'].sum())}")

display(manifest_output_df.head(30))
display(manifest_output_df[manifest_output_df["review_reason"].notna()].head(30))

## 4. Exportar outputs

Los archivos exportados son la base para validar antes de copiar al datastore y cargar al mosaico.

In [ ]:
summary_df = pd.DataFrame(
    [
        {"metric": "run_timestamp", "value": run_timestamp},
        {"metric": "audit_logic_version", "value": AUDIT_LOGIC_VERSION},
        {"metric": "input_folder", "value": PATH_INPUT_IMAGENES},
        {"metric": "sector_feature_class", "value": PATH_FC_INDICE_VUELOS_IMGS},
        {"metric": "sector_field", "value": SECTOR_FIELD_INDICE_VUELOS},
        {"metric": "sector_query", "value": QUERY_INDICE_VUELOS},
        {"metric": "input_files_count", "value": len(input_images_df)},
        {"metric": "ortho_images_count", "value": len(ortho_images_df)},
        {"metric": "ready_for_datastore_count", "value": int(manifest_output_df["ready_for_datastore"].sum())},
        {"metric": "review_required_count", "value": int((~manifest_output_df["ready_for_datastore"]).sum())},
        {"metric": "duplicate_expected_file_name_count", "value": int(manifest_output_df["duplicate_expected_file_name"].sum())},
    ]
)

for status, count in spatial_matches_df["spatial_status"].value_counts(dropna=False).items():
    summary_df.loc[len(summary_df)] = {"metric": f"spatial_status_{status}", "value": int(count)}

for status, count in manifest_df["rename_status"].value_counts(dropna=False).items():
    summary_df.loc[len(summary_df)] = {"metric": f"rename_status_{status}", "value": int(count)}

summary_csv = OUTPUT_DIR / "00_summary.csv"
input_csv = OUTPUT_DIR / "01_input_images.csv"
spatial_csv = OUTPUT_DIR / "02_spatial_matches.csv"
manifest_csv = OUTPUT_DIR / "03_manifest_carga_mosaico.csv"
ready_csv = OUTPUT_DIR / "04_ready_for_datastore.csv"
review_csv = OUTPUT_DIR / "05_review_required.csv"

summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
input_images_df.to_csv(input_csv, index=False, encoding="utf-8-sig")
spatial_matches_df.to_csv(spatial_csv, index=False, encoding="utf-8-sig")
manifest_output_df.to_csv(manifest_csv, index=False, encoding="utf-8-sig")
manifest_output_df[manifest_output_df["ready_for_datastore"]].to_csv(ready_csv, index=False, encoding="utf-8-sig")
manifest_output_df[~manifest_output_df["ready_for_datastore"]].to_csv(review_csv, index=False, encoding="utf-8-sig")

display(summary_df)
print("Outputs exportados en:", OUTPUT_DIR)
print("Manifest principal:", manifest_csv)